# DAF-05: One Row Per Day, With Tomorrow's Answer Attached

Asha has 554 raw 15-minute files for R K Puram. That's not a table a model
can learn from — it's not even a table yet, it's a pile of readings with
every pollutant mixed together.

Before any model or baseline can be scored (DAF-06, DAF-07), we need one row
per calendar day, and each row needs to carry the one thing a forecaster is
actually trying to predict: **tomorrow's 24-hour PM2.5 mean**.

The path is: 15-minute readings → hourly means → daily table → target.
Skipping straight from 15-minute to daily silently changes what "a day's
mean" means whenever hours have different numbers of readings.

## 1. Load every raw file for location 17 into one frame

554 files, one per station-day, all under
`data/raw/openaq/locationid=17/`. We read every one, concatenate, and never
write anything back into `data/raw/` — that folder is immutable.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
raw_dir = project_root / "data/raw/openaq/locationid=17"

files = sorted(raw_dir.rglob("*.csv.gz"))
print("Files found:", len(files))
print("First file:", files[0].name)
print("Last file:", files[-1].name)

frames = [pd.read_csv(f, compression="gzip") for f in files]
df = pd.concat(frames, ignore_index=True)

print("Files read into memory:", len(frames))
print("Combined shape (rows, cols):", df.shape)
print("Parameters present:", df["parameter"].unique())
df.head()


Files found: 554
First file: location-17-20250219.csv.gz
Last file: location-17-20260911.csv.gz
Files read into memory: 554
Combined shape (rows, cols): (467816, 9)
Parameters present: ['pm10' 'pm25' 'o3' 'no2' 'no' 'relativehumidity' 'temperature' 'so2' 'co'
 'wind_direction' 'nox' 'wind_speed']


,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,17,12234786,"R K Puram, Delhi - DPCC-3379481",2025-02-19T02:00:00+05:30,28.563262,77.186937,pm10,µg/m³,182.0
1,17,12234786,"R K Puram, Delhi - DPCC-3379481",2025-02-19T02:15:00+05:30,28.563262,77.186937,pm10,µg/m³,182.0
2,17,12234786,"R K Puram, Delhi - DPCC-3379481",2025-02-19T02:30:00+05:30,28.563262,77.186937,pm10,µg/m³,182.0
3,17,12234786,"R K Puram, Delhi - DPCC-3379481",2025-02-19T03:00:00+05:30,28.563262,77.186937,pm10,µg/m³,167.0
4,17,12234786,"R K Puram, Delhi - DPCC-3379481",2025-02-19T03:15:00+05:30,28.563262,77.186937,pm10,µg/m³,167.0


## 2. Keep PM2.5 only, and parse `datetime` as timezone-aware

Each file mixes multiple parameters (`pm25`, `pm10`, …) in the same rows.
We only care about `parameter == "pm25"`.

`datetime` already carries its own UTC+05:30 offset in the raw text. If we
parse it naively, pandas drops the offset and treats every timestamp as if
it were UTC — which shifts every day boundary by 5.5 hours. Parsing it
timezone-aware keeps IST midnight as IST midnight.


In [2]:
pm25 = df[df["parameter"] == "pm25"].copy()
print("Rows before filtering:", len(df))
print("Rows after keeping pm25 only:", len(pm25))

pm25["datetime"] = pd.to_datetime(pm25["datetime"])
print("Timezone on parsed datetime:", pm25["datetime"].dt.tz)

pm25 = pm25[["datetime", "value"]].sort_values("datetime").reset_index(drop=True)

duplicate_timestamps = pm25["datetime"].duplicated().sum()
print("Duplicate timestamps found:", duplicate_timestamps)

print("Date span:", pm25["datetime"].min(), "to", pm25["datetime"].max())
pm25.head()


Rows before filtering: 467816
Rows after keeping pm25 only: 44139
Timezone on parsed datetime: UTC+05:30
Duplicate timestamps found: 0
Date span: 2025-02-19 01:45:00+05:30 to 2026-09-12 00:00:00+05:30


,datetime,value
0,2025-02-19 01:45:00+05:30,123.0
1,2025-02-19 02:00:00+05:30,97.0
2,2025-02-19 02:15:00+05:30,97.0
3,2025-02-19 02:30:00+05:30,97.0
4,2025-02-19 02:45:00+05:30,97.0


## 3. Resample 15-minute readings to hourly means

### 😖 The problem

Right now `pm25` has one row every ~15 minutes — 4 readings per hour, but
not always exactly 4 (a sensor can drop a reading and leave a gap). D-002's
rule for "was this a good day" is written in **hours**: *a day only counts
if it has at least 18 hours with data*. We can't check that rule while the
data is still in 15-minute pieces — there's no `hours` column yet.

### 💡 The idea

Think of it like collapsing a Kafka topic's raw 15-minute events into
one hourly rollup record before anything downstream reads it — you don't
want every consumer re-deriving "was this hour covered" from raw events
themselves. So we do the rollup once, here, and everyone downstream uses it.

Concretely: take every reading that falls inside `09:00:00`–`09:59:59` and
average them into a single `09:00` value. Do that for every hour, across
the whole dataset.

### 🔍 What this actually is

`.resample("1h").mean()` is pandas' version of a `GROUP BY hour`. It walks
the whole time range in 1-hour steps and, for each step, averages whatever
readings fall inside it.

Two things to expect in the output:

- **An hour with zero readings still appears as a row** — its value is
  `NaN`, not missing entirely. That's on purpose: an hourly slot with no
  data is exactly what we need to count later ("hours" = hourly slots that
  are *not* `NaN`).
- **The timezone must survive.** Resampling groups by clock time, so if
  the index quietly lost its `+05:30` offset before this step, the hourly
  boundaries would land 5.5 hours off from real IST hours.

### What breaks if we skip this

If we averaged the raw 15-minute values straight into a daily mean, a day
that only has sensor data from midnight to 6 a.m. would look identical to
a day with full 24-hour coverage — both just produce "a mean". We'd have no
way to tell a trustworthy day from a mostly-missing one, which is exactly
what step 5 (marking valid/invalid days) needs to be able to do.


In [3]:
pm25_indexed = pm25.set_index("datetime")

hourly = pm25_indexed["value"].resample("1h").mean()

print("15-minute readings:", len(pm25_indexed))
print("Hourly slots after resampling:", len(hourly))
print("Hourly slots with no reading (NaN):", hourly.isna().sum())
print("Timezone preserved:", hourly.index.tz)

hourly.head()


15-minute readings: 44139
Hourly slots after resampling: 13680
Hourly slots with no reading (NaN): 1515
Timezone preserved: UTC+05:30


datetime
2025-02-19 01:00:00+05:30    123.0
2025-02-19 02:00:00+05:30     97.0
2025-02-19 03:00:00+05:30     88.0
2025-02-19 04:00:00+05:30    114.0
2025-02-19 05:00:00+05:30     67.0
Freq: h, Name: value, dtype: float64

## 4. Build the daily table: pm25_mean, hours, pm25_until_17

### 😖 The problem

`hourly` is still one row per hour — thousands of rows. We need one row
per **calendar day**, and each day needs to say two different things:
"what actually happened all day" and "what we knew by 6 p.m." A forecasting
model running at 6 p.m. can't see the rest of today yet — it hasn't
happened.

### 💡 The idea

Same rollup idea as before, one level up: group the hourly values by date
instead of by hour. But we need it twice, with different filters:

- once over **all 24 hours** of the day → the day's true mean
- once over **only hours 00:00–17:00** → what a model could have known if
  it ran at 6 p.m. that day

Think of it like a nightly batch job vs. an intraday snapshot: the nightly
job sees the whole day, the intraday snapshot only sees what happened
before it ran.

### 🔍 What this actually is

- `pm25_mean` = daily mean of `hourly`, over all hours.
- `hours` = how many of those 24 hourly slots are *not* `NaN` — this is the
  attendance count DAF-05's rule cares about.
- `pm25_until_17` = daily mean of `hourly`, but only over rows where the
  hour is between 0 and 17 inclusive.

### What breaks if we skip this

Without `pm25_until_17`, the only "today" feature available to a model
would be `pm25_mean` — which isn't complete until midnight. Using it as a
feature would leak information from hours that haven't happened yet at
prediction time.


In [4]:
# resample("D") groups the DatetimeIndex into calendar-day buckets (midnight to midnight,
# in whatever timezone the index already carries) — same idea as resample("1h") before,
# just a wider bucket. .mean() then collapses each bucket to a single number.
pm25_mean = hourly.resample("D").mean()

# .count() instead of .mean(): counts non-NaN hourly slots per day, ignoring NaN.
# This becomes the "hours" attendance number the >= 18 rule checks against.
hours = hourly.resample("D").count()
print("hours per day (first 10):\n", hours.head(10))

# Boolean mask on the DatetimeIndex's hour component: keeps only rows timestamped 00:00-17:00.
until_17 = hourly[hourly.index.hour <= 17]
# Same daily bucketing as pm25_mean, but applied to the already-filtered subset above.
pm25_until_17 = until_17.resample("D").mean()

# Combining three Series that share the same date index into one table, one column each.
daily = pd.DataFrame({
    "pm25_mean": pm25_mean,
    "hours": hours,
    "pm25_until_17": pm25_until_17,
})

print("Daily rows:", len(daily))
print("Duplicate dates:", daily.index.duplicated().sum())
print("Days with a fully-null pm25_until_17:", daily["pm25_until_17"].isna().sum())
print("hours summary:\n", daily["hours"].describe())

daily.head()

hours per day (first 10):
 datetime
2025-02-19 00:00:00+05:30    23
2025-02-20 00:00:00+05:30    24
2025-02-21 00:00:00+05:30    24
2025-02-22 00:00:00+05:30    24
2025-02-23 00:00:00+05:30    24
2025-02-24 00:00:00+05:30    24
2025-02-25 00:00:00+05:30    24
2025-02-26 00:00:00+05:30    24
2025-02-27 00:00:00+05:30    24
2025-02-28 00:00:00+05:30    24
Freq: D, Name: value, dtype: int64
Daily rows: 571
Duplicate dates: 0
Days with a fully-null pm25_until_17: 21
hours summary:
 count    571.000000
mean      21.304729
std        5.351336
min        0.000000
25%       22.000000
50%       24.000000
75%       24.000000
max       24.000000
Name: hours, dtype: float64


,pm25_mean,hours,pm25_until_17
datetime,,,
2025-02-19 00:00:00+05:30,78.782609,23,79.117647
2025-02-20 00:00:00+05:30,49.416667,24,49.611111
2025-02-21 00:00:00+05:30,90.375000,24,88.222222
2025-02-22 00:00:00+05:30,69.125000,24,69.777778
2025-02-23 00:00:00+05:30,74.000000,24,70.277778


## 5. Mark invalid days, but keep them for now

### 😖 The problem

Not every day in `daily` is trustworthy. Some days only had a handful of
hourly readings — maybe the sensor was offline for most of the day. A
`pm25_mean` built from 3 real hours and stretched to represent "the whole

day" is basically a guess wearing a daily average's clothes. If we let anumber that doesn't reflect real forecasting difficulty.

day like that become a **target**, we'd be training a model to chase noisebaseline scored against an unreliable target would report a fake error

instead of the real pollution pattern.A model trained on unreliable days would learn to predict noise, and a

Without `valid`, every day — reliable or not — would look equally usable.

### 💡 The idea

### What breaks if we skip this

D-002's rule already told us the bar: a day only counts if it has **at

least 18 out of 24 hours** with a real reading. So we don't need to decide  table to be looked up.

anything new here — we just need to apply that rule and label each row.  was valid, which only works if invalid rows are still sitting in the

- The next step (building `target`) needs to check whether **tomorrow**

Think of it like a request-success-rate SLA: a service that only answered  report).

3 out of 24 hourly health checks that day doesn't get to report "99% uptime"  else (that count is one of the four numbers this ticket asks us to

for the day — it gets flagged as unreliable, and anything built on top of- We want to **count** how many days are invalid before deciding anything

it (alerts, dashboards) should know that.

still keep every row. Two reasons to hold off on deleting:

### 🔍 What this actually isA new boolean column, `valid` = `hours >= 18`. Nothing gets deleted — we


In [5]:
daily["valid"] = daily["hours"] >= 18

print("Total days:", len(daily))
print("Valid days:", daily["valid"].sum())
print("Invalid days:", (~daily["valid"]).sum())
print("valid value counts:\n", daily["valid"].value_counts())

print("\nSample of invalid days:\n", daily[~daily["valid"]].head(10))

daily.head()


Total days: 571
Valid days: 488
Invalid days: 83
valid value counts:
 valid
True     488
False     83
Name: count, dtype: int64

Sample of invalid days:
                             pm25_mean  hours  pm25_until_17  valid
datetime                                                          
2025-03-02 00:00:00+05:30         NaN      0            NaN  False
2025-03-03 00:00:00+05:30   52.384615     13      49.857143  False
2025-03-15 00:00:00+05:30   34.058824     17      29.181818  False
2025-03-21 00:00:00+05:30   90.916667     12     104.125000  False
2025-04-07 00:00:00+05:30  112.352941     17     119.636364  False
2025-04-09 00:00:00+05:30  108.000000      7     108.000000  False
2025-04-10 00:00:00+05:30         NaN      0            NaN  False
2025-04-11 00:00:00+05:30   38.250000      4            NaN  False
2025-04-14 00:00:00+05:30  106.357143     14     106.357143  False
2025-04-15 00:00:00+05:30   66.933333     15      68.785714  False


,pm25_mean,hours,pm25_until_17,valid
datetime,,,,
2025-02-19 00:00:00+05:30,78.782609,23,79.117647,True
2025-02-20 00:00:00+05:30,49.416667,24,49.611111,True
2025-02-21 00:00:00+05:30,90.375000,24,88.222222,True
2025-02-22 00:00:00+05:30,69.125000,24,69.777778,True
2025-02-23 00:00:00+05:30,74.000000,24,70.277778,True


## 6. Build `target`: tomorrow's pm25_mean — by date, not by row

### 😖 The problem

We have `daily`, indexed by date, with `pm25_mean` and `valid`. What's
still missing is the one thing a forecaster actually needs: for each day,
"what did PM2.5 turn out to be **the next day**?" That's `target` — without
it, there's nothing for a model to learn to predict, and nothing to score
a baseline against.

### 💡 The idea

In plain terms: walk down to the next row and copy its `pm25_mean` up as
today's `target`. The trap is that our date index has gaps wherever a day
is missing entirely — a plain shift-by-row would then quietly pair, say,
Monday with Wednesday's value instead of the (missing) Tuesday's. We avoid
that by first filling in every missing date as its own row, so "next row"
and "next calendar day" always mean the same thing.

We also don't want a target built from an unreliable day — so a day only
gets a `target` when the next day was itself `valid`.

### 🔍 A tiny 5-row example

Nothing gets dropped in this step — `daily` already has `pm25_mean`,
`hours`, `pm25_until_17`, and `valid` from steps 4 and 5. We only ever
**add** a `target` column on top; every existing column stays exactly as
it was.

Say `daily` looked like this before we touch anything (Jan 3 is missing
entirely — sensor was down all day, so it never became a row at all):

| date | pm25_mean | hours | pm25_until_17 | valid |
|---|---|---|---|---|
| Jan 1 | 40 | 24 | 38 | True |
| Jan 2 | 55 | 22 | 50 | True |
| Jan 4 | 90 | 21 | 88 | True |
| Jan 5 | 70 | 10 | 65 | **False** ← only 10 hours had a reading that day (< 18) |
| Jan 6 | 30 | 24 | 28 | True |

**Step A — reindex (fill the gap):** Jan 3 gets inserted as a brand new
row. All of its columns are `NaN` because nothing was ever computed for
it, except `valid`, which we explicitly set to `False`:

| date | pm25_mean | hours | pm25_until_17 | valid |
|---|---|---|---|---|
| Jan 1 | 40 | 24 | 38 | True |
| Jan 2 | 55 | 22 | 50 | True |
| Jan 3 | NaN | NaN | NaN | **False** ← 0 hours of data, sensor was down all day |
| Jan 4 | 90 | 21 | 88 | True |
| Jan 5 | 70 | 10 | 65 | **False** ← only 10 hours had a reading that day (< 18) |
| Jan 6 | 30 | 24 | 28 | True |

**Step B — shift `pm25_mean` up by one row** to make `target` (every row
now looks at the row directly below it, which is guaranteed to be the
actual next calendar day). `hours` and `pm25_until_17` are untouched —
this step only ever reads `pm25_mean` to build the new `target` column:

| date | pm25_mean | hours | pm25_until_17 | valid | target (before the valid check) |
|---|---|---|---|---|---|
| Jan 1 | 40 | 24 | 38 | True | 55 *(copied from Jan 2)* |
| Jan 2 | 55 | 22 | 50 | True | NaN *(copied from Jan 3 — Jan 3 has no data)* |
| Jan 3 | NaN | NaN | NaN | False | 90 *(copied from Jan 4)* |
| Jan 4 | 90 | 21 | 88 | True | 70 *(copied from Jan 5)* |
| Jan 5 | 70 | 10 | 65 | False | 30 *(copied from Jan 6)* |
| Jan 6 | 30 | 24 | 28 | True | NaN *(no Jan 7 row exists)* |

**Step C — blank out targets where tomorrow wasn't `valid`:** Jan 4's
target came from Jan 5, but Jan 5 is `valid = False` — so that target gets
wiped to `NaN` too. This is the final shape of `daily`, with every original
column still intact and `target` added at the end:

| date | pm25_mean | hours | pm25_until_17 | valid | target (final) |
|---|---|---|---|---|---|
| Jan 1 | 40 | 24 | 38 | True | 55 |
| Jan 2 | 55 | 22 | 50 | True | NaN |
| Jan 3 | NaN | NaN | NaN | False | 90 |
| Jan 4 | 90 | 21 | 88 | True | **NaN** *(Jan 5 was invalid)* |
| Jan 5 | 70 | 10 | 65 | False | 30 |
| Jan 6 | 30 | 24 | 28 | True | NaN |

Notice Jan 1's target is correctly Jan 2's value, not Jan 3's or Jan 4's —
that's only true because we reindexed first. Without Step A, `shift(-1)`
on the original 5-row table (no gap) would have wrongly given Jan 2 a
target of 90 (Jan 4's value), skipping right over the missing Jan 3.





In [6]:
rows_before = len(daily)
print("Rows before reindexing:", rows_before)

# print(daily.shape)
# print(daily.index.min())
# print (daily)

# date_range fills in every date between first and last, even ones daily has no row for.
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D", tz=daily.index.tz)

print("Full Range ",full_range)

# reindex snaps daily onto that gap-free list: existing dates keep their row,
# dates that were missing get inserted brand new, filled with NaN in every column.
daily = daily.reindex(full_range)

# The newly-inserted rows have valid = NaN (nothing computed it) -> treat "no data at all" as invalid.
daily["valid"] = daily["valid"].fillna(False)

print("Rows after reindexing (gap-free):", len(daily))
print("Missing calendar dates that got added:", len(daily) - rows_before)

# shift(-1) is now safe: because the index has no gaps, "next row" == "next calendar date".
daily["target"] = daily["pm25_mean"].shift(-1)
print(daily)

# Shift valid the same way, to line up "was tomorrow valid" against today's row.
next_day_valid = daily["valid"].shift(-1).fillna(False)

# .where(cond) keeps target only where next_day_valid is True, else overwrites it with NaN.
daily["target"] = daily["target"].where(next_day_valid)

print("Rows with a usable target:", daily["target"].notna().sum())

missing_next_day_example = daily[daily["pm25_mean"].notna() & daily["target"].isna()].head(1)
print("\nExample row where target is empty:\n", missing_next_day_example)

daily.head(10)


Rows before reindexing: 571
Full Range  DatetimeIndex(['2025-02-19 00:00:00+05:30', '2025-02-20 00:00:00+05:30',
               '2025-02-21 00:00:00+05:30', '2025-02-22 00:00:00+05:30',
               '2025-02-23 00:00:00+05:30', '2025-02-24 00:00:00+05:30',
               '2025-02-25 00:00:00+05:30', '2025-02-26 00:00:00+05:30',
               '2025-02-27 00:00:00+05:30', '2025-02-28 00:00:00+05:30',
               ...
               '2026-09-03 00:00:00+05:30', '2026-09-04 00:00:00+05:30',
               '2026-09-05 00:00:00+05:30', '2026-09-06 00:00:00+05:30',
               '2026-09-07 00:00:00+05:30', '2026-09-08 00:00:00+05:30',
               '2026-09-09 00:00:00+05:30', '2026-09-10 00:00:00+05:30',
               '2026-09-11 00:00:00+05:30', '2026-09-12 00:00:00+05:30'],
              dtype='datetime64[ns, UTC+05:30]', length=571, freq='D')
Rows after reindexing (gap-free): 571
Missing calendar dates that got added: 0
                           pm25_mean  hours  pm25_until_17  

/var/folders/07/ymtnct793nj_13sf3p9t0lqh0000gn/T/ipykernel_96814/3846804771.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  next_day_valid = daily["valid"].shift(-1).fillna(False)


,pm25_mean,hours,pm25_until_17,valid,target
2025-02-19 00:00:00+05:30,78.782609,23,79.117647,True,49.416667
2025-02-20 00:00:00+05:30,49.416667,24,49.611111,True,90.375000
2025-02-21 00:00:00+05:30,90.375000,24,88.222222,True,69.125000
2025-02-22 00:00:00+05:30,69.125000,24,69.777778,True,74.000000
2025-02-23 00:00:00+05:30,74.000000,24,70.277778,True,101.166667
2025-02-24 00:00:00+05:30,101.166667,24,102.666667,True,101.958333
2025-02-25 00:00:00+05:30,101.958333,24,98.666667,True,118.166667
2025-02-26 00:00:00+05:30,118.166667,24,114.277778,True,68.000000
2025-02-27 00:00:00+05:30,68.000000,24,75.111111,True,56.875000
2025-02-28 00:00:00+05:30,56.875000,24,56.666667,True,59.217391


## 7. Save, summarise, and visualise the daily table

We now have the finished DAF-05 table. Each row represents one calendar day.
The existing columns are kept, and the new `target` column represents the
next day's PM2.5 mean.

For the visual check, we focus on the first 90 days so the daily pattern is
easy to read. You can hover over any point or bar to see the exact date and
values.

- The PM2.5 chart shows the measured daily mean and the next-day target.
- The coverage chart shows how many of the possible 24 hourly buckets had data.
  For example, `24` means all hours had data, while `13` means only 13 hours
  had data and the day is invalid because it is below the 18-hour rule.
- Green bars are valid days; red bars are invalid days.

The raw files remain untouched. We save the processed table under
`data/interim/` and print the four DAF-05 counts.


In [8]:
interim_dir = project_root / "data/interim"
interim_dir.mkdir(parents=True, exist_ok=True)

output_path = interim_dir / "daily_17.csv"
daily.to_csv(output_path, index_label="date")

print("Saved daily table to:", output_path)
print("Days in span:", len(daily))
print("Valid days:", daily["valid"].sum())
print("Invalid days:", (~daily["valid"]).sum())
print("Rows with a usable target:", daily["target"].notna().sum())
print("Output shape:", daily.shape)
print("Output columns:", daily.columns.tolist())

print("\nSaved table preview:")
print(daily.head())

import plotly.graph_objects as go

# Use the first 90 days for a readable visual while saving the complete table above.
plot_data = daily.reset_index(names="date")
focus_data = plot_data.head(90).copy()
focus_data["valid_label"] = focus_data["valid"].map({True: "Valid", False: "Invalid"})
focus_data["coverage_color"] = focus_data["valid"].map({True: "Valid day", False: "Invalid day"})

print("\nVisual window:", focus_data["date"].min(), "to", focus_data["date"].max())

pm25_figure = go.Figure()
pm25_figure.add_trace(go.Scatter(
    x=focus_data["date"],
    y=focus_data["pm25_mean"],
    mode="lines+markers",
    name="Daily PM2.5 mean",
    customdata=focus_data[["hours", "valid_label", "target"]],
    hovertemplate=(
        "Date: %{x|%Y-%m-%d}<br>"
        "Daily PM2.5 mean: %{y:.2f} µg/m³<br>"
        "Hours with data: %{customdata[0]} / 24<br>"
        "Valid: %{customdata[1]}<br>"
        "Next-day target: %{customdata[2]:.2f} µg/m³<extra></extra>"
    ),
))
pm25_figure.add_trace(go.Scatter(
    x=focus_data["date"],
    y=focus_data["target"],
    mode="lines+markers",
    name="Next-day target",
    hovertemplate=(
        "Date: %{x|%Y-%m-%d}<br>"
        "Next-day target: %{y:.2f} µg/m³<extra></extra>"
    ),
))
pm25_figure.update_layout(
    title="First 90 Days: Daily PM2.5 and Next-Day Target",
    xaxis_title="Date",
    yaxis_title="PM2.5 (µg/m³)",
    hovermode="x unified",
)
pm25_figure.show()

coverage_figure = go.Figure()
coverage_figure.add_trace(go.Bar(
    x=focus_data["date"],
    y=focus_data["hours"],
    name="Hourly buckets with data",
    marker_color=focus_data["valid"].map({True: "seagreen", False: "crimson"}),
    customdata=focus_data[["pm25_mean", "valid_label", "target"]],
    hovertemplate=(
        "Date: %{x|%Y-%m-%d}<br>"
        "Hours with data: %{y} / 24<br>"
        "Valid: %{customdata[1]}<br>"
        "Daily PM2.5 mean: %{customdata[0]:.2f} µg/m³<br>"
        "Next-day target: %{customdata[2]:.2f} µg/m³<extra></extra>"
    ),
))
coverage_figure.add_hline(
    y=18,
    line_dash="dash",
    annotation_text="18-hour validity threshold",
)
coverage_figure.update_layout(
    title="First 90 Days: How Many Hours Were Available Each Day?",
    xaxis_title="Date",
    yaxis_title="Hours with PM2.5 data (out of 24)",
    yaxis_range=[0, 25],
    showlegend=False,
)
coverage_figure.show()

print("Days plotted:", len(focus_data))
print("Days with a target in visual window:", focus_data["target"].notna().sum())
print("Lowest coverage in visual window:", focus_data["hours"].min(), "/ 24 hours")
print("Highest coverage in visual window:", focus_data["hours"].max(), "/ 24 hours")


Saved daily table to: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/data/interim/daily_17.csv
Days in span: 571
Valid days: 488
Invalid days: 83
Rows with a usable target: 487
Output shape: (571, 5)
Output columns: ['pm25_mean', 'hours', 'pm25_until_17', 'valid', 'target']

Saved table preview:
                           pm25_mean  hours  pm25_until_17  valid      target
2025-02-19 00:00:00+05:30  78.782609     23      79.117647   True   49.416667
2025-02-20 00:00:00+05:30  49.416667     24      49.611111   True   90.375000
2025-02-21 00:00:00+05:30  90.375000     24      88.222222   True   69.125000
2025-02-22 00:00:00+05:30  69.125000     24      69.777778   True   74.000000
2025-02-23 00:00:00+05:30  74.000000     24      70.277778   True  101.166667

Visual window: 2025-02-19 00:00:00+05:30 to 2025-05-19 00:00:00+05:30


Days plotted: 90
Days with a target in visual window: 74
Lowest coverage in visual window: 0 / 24 hours
Highest coverage in visual window: 24 / 24 hours


## 8. Final validation of the DAF-05 table

The daily table is built and saved. Before we move to DAF-06, we do a final
check that the table obeys the ticket rules:

- one row per date and no duplicate dates
- all required columns are present
- invalid days have fewer than 18 hours
- a usable target exists only when the next day is valid
- the raw archive was not changed by this notebook

The summary chart compares the number of valid and invalid days, then the
number of rows with and without a usable target. This gives us a quick visual
check of how much data is ready for baseline scoring.


In [9]:
required_columns = {
    "pm25_mean",
    "hours",
    "pm25_until_17",
    "valid",
    "target",
}

assert daily.index.is_unique, "Daily table contains duplicate dates"
assert required_columns.issubset(daily.columns), "A required column is missing"
assert (daily.loc[~daily["valid"], "hours"] < 18).all(), "Invalid day has 18 or more hours"

next_day_is_valid = daily["valid"].shift(-1).fillna(False)
usable_target_rows = daily["target"].notna()
assert usable_target_rows.equals(daily["target"].notna() & next_day_is_valid), (
    "A target exists for a row whose next day is invalid"
)

raw_files_after = sorted(raw_dir.rglob("*.csv.gz"))
assert len(raw_files_after) == len(files), "Raw file count changed"

print("Duplicate dates:", daily.index.duplicated().sum())
print("Required columns present:", required_columns.issubset(daily.columns))
print("Invalid days below 18 hours:", (daily.loc[~daily["valid"], "hours"] < 18).all())
print("Raw file count unchanged:", len(raw_files_after) == len(files))
print("All DAF-05 validation checks passed")

validation_labels = ["Valid days", "Invalid days", "Usable targets", "Missing targets"]
validation_values = [
    int(daily["valid"].sum()),
    int((~daily["valid"]).sum()),
    int(daily["target"].notna().sum()),
    int(daily["target"].isna().sum()),
]

validation_figure = go.Figure(
    go.Bar(
        x=validation_labels,
        y=validation_values,
        text=validation_values,
        textposition="auto",
        marker_color=["seagreen", "crimson", "steelblue", "lightgray"],
        hovertemplate="%{x}: %{y} days<extra></extra>",
    )
)
validation_figure.update_layout(
    title="DAF-05 Data Readiness Summary",
    xaxis_title="Validation category",
    yaxis_title="Number of days",
    showlegend=False,
)
validation_figure.show()


Duplicate dates: 0
Required columns present: True
Invalid days below 18 hours: True
Raw file count unchanged: True
All DAF-05 validation checks passed


/var/folders/07/ymtnct793nj_13sf3p9t0lqh0000gn/T/ipykernel_96814/3634938761.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  next_day_is_valid = daily["valid"].shift(-1).fillna(False)
